# Session 2: Prompt Engineering for Agentic Behaviors (Instructor Notebook — Full Solutions)

## Learning Objectives

By the end of this session, students will be able to:

1. **Craft effective system prompts** that define agent personas, capabilities, and constraints
2. **Implement Chain-of-Thought (CoT) and ReAct prompting** patterns to improve reasoning quality
3. **Generate structured outputs** (JSON mode) for reliable downstream processing
4. **Manage multi-turn conversations** with context accumulation and summarization strategies

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# Imports and Setup
import openai
import json
import os

# Ensure your OPENAI_API_KEY is set in your environment variables
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

client = openai.OpenAI()
print("Setup complete. OpenAI client initialized.")

---
## Demos

Walk through these demos with the students. Run each cell and discuss the outputs.

### Demo 1: Zero-Shot vs. Few-Shot Prompting

**Zero-shot** prompting asks the model to perform a task with no examples.  
**Few-shot** prompting provides a handful of examples so the model can learn the expected format and behavior.

We will compare both approaches on a **client feedback classification** task — categorizing engagement survey responses as positive, negative, or neutral.

**Instructor Note:** Emphasize that few-shot prompting is one of the simplest yet most effective techniques for controlling output format. Point out the difference in verbosity between the two responses.

In [ ]:
# Demo 1: Zero-Shot vs. Few-Shot Prompting

# Zero-shot: no examples provided
zero_shot_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Classify the sentiment of this client engagement feedback as positive, negative, or neutral: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}]
)
print("Zero-shot:", zero_shot_response.choices[0].message.content)

print("\n" + "="*60 + "\n")

# Few-shot: provide labeled examples from past McKinsey engagement surveys
few_shot_messages = [
    {"role": "system", "content": "You are a McKinsey client feedback classifier. Respond with only: POSITIVE, NEGATIVE, or NEUTRAL."},
    {"role": "user", "content": "Feedback: 'The engagement exceeded our expectations — the team was exceptional and the insights transformed our strategy.'"},
    {"role": "assistant", "content": "POSITIVE"},
    {"role": "user", "content": "Feedback: 'Disappointed with the engagement. Deliverables were late and the analysis did not address our core issues.'"},
    {"role": "assistant", "content": "NEGATIVE"},
    {"role": "user", "content": "Feedback: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}
]
few_shot_response = client.chat.completions.create(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=few_shot_messages)
print("Few-shot:", few_shot_response.choices[0].message.content)

**Observation:** Notice how the zero-shot response may include extra explanation, while the few-shot response follows the exact format demonstrated in the examples.

### Demo 2: Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting encourages the model to show its reasoning process step by step, which improves accuracy on math, logic, and multi-step problems.

**Instructor Note:** The key insight here is that LLMs perform better when they "think out loud." This mirrors McKinsey's structured problem-solving — breaking a problem into steps before jumping to the answer. CoT is especially valuable for market sizing and financial analysis.

In [ ]:
# Demo 2: Chain-of-Thought Prompting

problem = "A McKinsey client has 1,200 retail stores. They plan to close 20% of underperforming locations and increase revenue per remaining store by 15%. If current average revenue per store is $2M, what will total revenue be after the restructuring?"

# Without CoT
response_no_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": problem}]
)
print("Without CoT:")
print(response_no_cot.choices[0].message.content)

print("\n" + "="*60 + "\n")

# With explicit CoT — mirroring McKinsey structured problem-solving
cot_prompt = f"{problem}\n\nLet's solve this step by step:\n1. First, calculate how many stores will be closed\n2. Then, find the number of remaining stores\n3. Calculate the new revenue per store after the 15% increase\n4. Compute total projected revenue"
response_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": cot_prompt}]
)
print("With CoT:")
print(response_cot.choices[0].message.content)

**Observation:** The CoT version explicitly walks through each calculation, making it easier to verify correctness and debug any errors in reasoning.

### Demo 3: Role-Based Prompting for Agentic Personas

By changing the system prompt, we can create entirely different McKinsey practice-area personas that approach the same client question from different angles. This is foundational for building specialized agents in multi-agent consulting systems.

**Instructor Note:** Discuss how multi-agent systems leverage this — each agent can be a different practice area expert working on the same client problem, mirroring how McKinsey staffs cross-functional teams.

In [ ]:
# Demo 3: Role-Based Prompting for Agentic Personas

personas = {
    "McKinsey Growth Strategy Lead": "You are a McKinsey partner in Growth & Innovation. You approach every question by identifying growth levers, market opportunities, and revenue acceleration strategies. Use frameworks like the Three Horizons of Growth.",
    "McKinsey Risk & Compliance Expert": "You are a McKinsey senior expert in Risk & Resilience. You evaluate everything through the lens of regulatory risk, compliance requirements, operational resilience, and enterprise risk management.",
    "McKinsey Organization & People Lead": "You are a McKinsey partner in People & Organizational Performance. You think about talent strategy, organizational design, change management, and leadership development. Focus on the human side of transformation."
}
question = "Our banking client is considering launching a digital-only subsidiary to compete with fintechs."

for role, system_prompt in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
    )
    print(f"\n{'='*50}\n{role}:\n{response.choices[0].message.content}")

**Observation:** Each persona focuses on different aspects of the same question -- data metrics, security risks, or user/business impact. This demonstrates how system prompts shape agent behavior.

### Demo 4: Structured Output Generation (JSON Mode)

When building agentic systems for McKinsey workflows, we often need the LLM output to be machine-readable — for example, extracting structured client data from meeting notes or parsing engagement details from unstructured text. JSON mode ensures the response is valid JSON, making downstream processing reliable.

**Instructor Note:** Highlight that `response_format={"type": "json_object"}` is an API-level guarantee. Without it, the model might produce mostly-valid JSON but occasionally break format — unacceptable in production consulting tools.

In [ ]:
# Demo 4: Structured Output Generation (JSON Mode)

text = "Sarah Chen is the CFO of Meridian Healthcare, a $3.2B revenue hospital network based in Chicago. She has 18 years of experience in healthcare finance and previously led M&A at Deloitte. Her priorities for the McKinsey engagement are cost optimization, revenue cycle management, and evaluating two potential acquisition targets."

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "Extract the client executive's information and return it as a JSON object with keys: name, title, company, company_revenue, location, experience_years, previous_employer, engagement_priorities (as a list)."},
        {"role": "user", "content": text}
    ],
    response_format={"type": "json_object"}
)

parsed = json.loads(response.choices[0].message.content)
print("Parsed JSON output:")
print(json.dumps(parsed, indent=2))

# Demonstrate that we can access individual fields programmatically
print(f"\nClient Executive: {parsed.get('name')}, {parsed.get('title')}")
print(f"Engagement Priorities: {', '.join(parsed.get('engagement_priorities', []))}")

**Observation:** With `response_format={"type": "json_object"}`, the output is guaranteed to be valid JSON, which we can reliably parse and use in code.

### Demo 5: Multi-Turn Conversation Management

Consulting agents need to maintain context across multiple exchanges — remembering the client industry, engagement scope, and prior recommendations. This demo shows how to build a conversation manager that accumulates context, mirroring how a McKinsey engagement progresses through multiple discussions.

**Instructor Note:** Discuss the trade-off between keeping full history (better context) and truncating/summarizing (staying within token limits). In long-running engagements with extensive documentation, you need a strategy for managing context window size.

In [ ]:
# Demo 5: Multi-Turn Conversation Management

class ConversationManager:
    def __init__(self, system_prompt, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]
        self.client = openai.OpenAI()
    
    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        )
        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg
    
    def get_history_length(self):
        return len(self.messages)

# Demo: Multi-turn McKinsey engagement planning conversation
conv = ConversationManager("You are a McKinsey engagement planning assistant. Help consultants scope and plan client engagements. Be structured, use MECE principles, and provide actionable next steps.")

print("User: We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%.")
print("Assistant:", conv.send("We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%."))

print("\n" + "-"*50)
print("\nUser: What data should we request from the client in the first week?")
print("Assistant:", conv.send("What data should we request from the client in the first week?"))

print("\n" + "-"*50)
print("\nUser: Can you summarize our engagement scope so far?")
print("Assistant:", conv.send("Can you summarize our engagement scope so far?"))

print(f"\nConversation history: {conv.get_history_length()} messages")

**Observation:** The assistant remembers context from earlier turns -- it knows we are discussing Japan without being reminded. The conversation history grows with each exchange.

### Demo 6: Structured Outputs with LangChain Prompt Templates

LangChain provides a higher-level abstraction for prompt engineering. Its `ChatPromptTemplate` supports parameterized prompts, and `OutputParser` classes enforce structured output formats (JSON, lists, etc.) — reducing boilerplate and improving reliability.

**Instructor Note:** Compare this with Demo 4 (raw JSON mode). LangChain templates are reusable, composable, and integrate with chains/agents. This is the industry-standard approach for production-grade prompt workflows.

In [ ]:
# Demo 6: Structured Outputs with LangChain Prompt Templates

# Install if needed: pip install langchain langchain-openai
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.output_parsers import JsonOutputParser

# Step 1: Initialize the LangChain LLM wrapper
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# --- Part A: ChatPromptTemplate with variables ---
print("PART A: LangChain ChatPromptTemplate for McKinsey Analysis")
print("=" * 60)

# Reusable template for industry analysis across different client engagements
analysis_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey {practice_area} expert. Provide structured, partner-ready analysis."),
    ("user", "Analyze the following for our client engagement: {topic}\n\nProvide your analysis in exactly 3 bullet points with strategic implications.")
])

prompt_value = analysis_template.invoke({"practice_area": "Operations", "topic": "supply chain resilience in the semiconductor industry"})
response = llm.invoke(prompt_value)
print(f"Practice: Operations | Topic: semiconductor supply chain")
print(response.content)

print()
prompt_value2 = analysis_template.invoke({"practice_area": "Strategy & Corporate Finance", "topic": "consolidation trends in European banking"})
response2 = llm.invoke(prompt_value2)
print(f"Practice: Strategy | Topic: European banking consolidation")
print(response2.content)

# --- Part B: StructuredOutputParser for McKinsey deliverables ---
print("\n" + "=" * 60)
print("PART B: StructuredOutputParser — McKinsey Engagement Summary")
print("=" * 60)

response_schemas = [
    ResponseSchema(name="executive_summary", description="A 1-2 sentence executive summary suitable for a CEO briefing"),
    ResponseSchema(name="key_findings", description="A list of 3 key findings as a JSON array of strings"),
    ResponseSchema(name="strategic_priority", description="The single most important strategic priority: growth, cost_optimization, digital_transformation, or M&A"),
    ResponseSchema(name="confidence", description="Confidence score from 0.0 to 1.0")
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()
print(f"Format instructions (auto-generated):\n{format_instructions}\n")

structured_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey engagement manager preparing a structured deliverable."),
    ("user", "Analyze the strategic landscape for: {topic}\n\n{format_instructions}")
])

prompt = structured_template.invoke({
    "topic": "a mid-size US retailer considering an omnichannel transformation",
    "format_instructions": format_instructions
})

response = llm.invoke(prompt)
parsed_output = output_parser.parse(response.content)

print("Parsed structured output:")
for key, value in parsed_output.items():
    print(f"  {key}: {value}")

# --- Part C: LangChain Chain (Prompt | LLM | Parser) ---
print("\n" + "=" * 60)
print("PART C: LangChain Chain — Prompt | LLM | Parser")
print("=" * 60)

chain = structured_template | llm | output_parser

result = chain.invoke({
    "topic": "a healthcare provider evaluating AI-powered diagnostics for radiology",
    "format_instructions": format_instructions
})

print("Chain output (directly parsed):")
print(json.dumps(result, indent=2))

# Task 1 Solution: Design Effective System Prompts for a McKinsey Research Agent

research_agent_prompt = """You are a McKinsey Industry Research Agent specializing in analyzing market dynamics, competitive landscapes, and strategic opportunities for client engagements.

## Your Approach:
1. Break down the research question into MECE sub-questions
2. Analyze each dimension with data-driven reasoning
3. Synthesize findings into a structured, partner-ready response

## Output Format:
Always structure your response as:
- **Executive Summary**: 2-3 sentence overview suitable for a CEO briefing
- **Key Findings**: Bullet points of main discoveries with quantitative evidence where possible
- **Strategic Implications**: What this means for the client's competitive position
- **Confidence Level**: High/Medium/Low with justification
- **Data Gaps**: What additional data would strengthen the analysis

## Guidelines:
- If uncertain, explicitly state your confidence level and flag assumptions
- Distinguish between facts, estimates, and inferences
- Use McKinsey-standard language and frameworks (Porter's Five Forces, value chain analysis, etc.)
- Provide balanced perspectives — always consider the 'so what?' for the client"""

def ask_research_agent(question):
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": research_agent_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=800
    )
    result = response.choices[0].message.content
    print(result)
    return result

ask_research_agent("What are the key growth drivers and competitive dynamics in the global electric vehicle battery market?")

### Task 2: Implement ReAct-Style Prompting (Reason + Act)

**Goal:** Implement the ReAct pattern where the model interleaves reasoning (Thought) with actions (Action) and observations (Observation) — applied to a McKinsey due diligence scenario.

**Key Teaching Points:**
- ReAct is a foundational pattern for agentic AI — it mirrors how McKinsey consultants structure their analysis: hypothesize, gather data, refine.
- In this demo, the LLM simulates both the reasoning and the tool results. In a real system, Actions would trigger actual data lookups.
- The explicit format makes the reasoning chain transparent and reviewable — critical for client-facing work.

In [ ]:
# Task 2 Solution: Implement ReAct-Style Prompting (Reason + Act)

def react_agent(question):
    react_prompt = """You are a McKinsey due diligence ReAct agent that solves problems by interleaving reasoning and actions.

Available Actions:
- MarketData[query]: Look up market size, growth rates, and industry data
- FinancialAnalysis[expression]: Perform financial calculations (margins, multiples, CAGR)
- CompetitorLookup[company]: Research competitor positioning and market share

Follow this EXACT format for each step:
Thought: [Your hypothesis or reasoning about what data you need next]
Action: [ActionName][input]
Observation: [What the data shows]

Continue until you have enough information, then provide:
Final Answer: [Your complete, structured recommendation]

Always show your complete reasoning chain — this is a client-facing deliverable."""

    client = openai.OpenAI()
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": react_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500"))
    )
    result = response.choices[0].message.content
    print(result)
    return result

react_agent("Is a $200M acquisition of a mid-size European logistics company a good deal for our private equity client, given the current market conditions?")

### Task 3: Create a Reusable Prompt Template Library

**Goal:** Build a `PromptTemplate` class that manages parameterized prompts with validation — tailored for common McKinsey deliverable types.

**Key Teaching Points:**
- Template libraries reduce duplication and enforce consistency across McKinsey engagement outputs.
- Validation prevents runtime errors from missing variables.
- This pattern is used in frameworks like LangChain (`PromptTemplate`) and is an industry best practice for scaling consulting AI tools.

In [ ]:
# Task 3 Solution: Create a Reusable Prompt Template Library

import re

class PromptTemplate:
    def __init__(self, name, template, description=""):
        self.name = name
        self.template = template
        self.description = description
        self.variables = re.findall(r'\{(\w+)\}', template)
    
    def format(self, **kwargs):
        self.validate(**kwargs)
        return self.template.format(**kwargs)
    
    def validate(self, **kwargs):
        missing = [v for v in self.variables if v not in kwargs]
        if missing:
            raise ValueError(f"Missing required variables: {missing}")
        return True

# McKinsey consulting template library
market_sizing_template = PromptTemplate(
    name="market_sizing",
    template="Estimate the total addressable market (TAM) for {product_category} in {geography}. Break down using a {approach} approach and provide your assumptions in {length} bullet points.",
    description="Market sizing analysis for client engagements"
)

executive_briefing_template = PromptTemplate(
    name="executive_briefing",
    template="Prepare a {length}-sentence executive briefing on {topic} for a {industry} client CEO. Focus on strategic implications and actionable recommendations.",
    description="Executive briefing for C-suite client meetings"
)

competitive_analysis_template = PromptTemplate(
    name="competitive_analysis",
    template="Analyze the competitive landscape for {company} in the {industry} sector. Identify the top {num_competitors} competitors and evaluate them on: {evaluation_criteria}.",
    description="Competitive landscape analysis for strategy engagements"
)

# Demo usage
prompt = market_sizing_template.format(
    product_category="AI-powered diagnostic tools",
    geography="North America",
    approach="top-down",
    length="5"
)
print(f"Generated prompt:\n{prompt}")
print(f"\nTemplate variables: {market_sizing_template.variables}")

# Show validation error
print("\n--- Testing validation ---")
try:
    market_sizing_template.format(product_category="AI tools")
except ValueError as e:
    print(f"Validation error (expected): {e}")

### Task 4: Build a Multi-Step Reasoning Agent with Tool Descriptions

**Goal:** Build a complete agentic loop where the LLM acts as a McKinsey analyst, deciding which consulting tool to call, receiving simulated results, and iterating until it has a final recommendation.

**Key Teaching Points:**
- This is the core pattern of tool-using agents: perceive -> reason -> act -> observe -> repeat.
- JSON mode is essential so the agent's tool calls are machine-parseable.
- The `simulate_tool` method stands in for real integrations (market databases, financial models, CRM systems).
- The loop has a max-step limit to prevent infinite loops — an important safety measure for production systems.

In [ ]:
# Task 4 Solution: Build a Multi-Step Reasoning Agent with Tool Descriptions

tools = [
    {"name": "market_database", "description": "Look up market size, growth rates, and industry trends", "parameters": "query (string)"},
    {"name": "financial_model", "description": "Run financial calculations: DCF, multiples, CAGR, margin analysis", "parameters": "expression (string)"},
    {"name": "benchmarking", "description": "Compare client KPIs against industry benchmarks", "parameters": "metric and industry (string)"}
]

class ToolAgent:
    def __init__(self):
        self.client = openai.OpenAI()
        tool_descriptions = "\n".join([f"- {t['name']}: {t['description']} (params: {t['parameters']})" for t in tools])
        self.system_prompt = f"""You are a McKinsey analyst agent with access to the following consulting tools:

{tool_descriptions}

When you need to use a tool, respond with JSON:
{{"thought": "your analytical reasoning", "tool": "tool_name", "input": "tool_input"}}

When you have enough information to provide a recommendation, respond with JSON:
{{"thought": "final synthesis", "tool": "final_answer", "input": "your complete recommendation"}}

Process one analytical step at a time. Think like a McKinsey consultant — be structured and hypothesis-driven."""
    
    def simulate_tool(self, tool_name, tool_input):
        simulated = {
            "market_database": f"Market data for '{tool_input}': Market size $45B, growing at 8.2% CAGR, top 3 players hold 55% share",
            "financial_model": f"Financial analysis for '{tool_input}': Implied EV/EBITDA multiple of 12.5x, IRR of 18.3% over 5-year hold period",
            "benchmarking": f"Benchmark for '{tool_input}': Client at 75th percentile vs peers, top quartile in revenue growth but below median in EBITDA margin"
        }
        return simulated.get(tool_name, "Tool not found")
    
    def process(self, question):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": question}
        ]
        
        for step in range(5):  # Max 5 analytical steps
            response = self.client.chat.completions.create(
                model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
                messages=messages,
                response_format={"type": "json_object"},
                max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
            )
            result = json.loads(response.choices[0].message.content)
            print(f"\nStep {step + 1}:")
            print(f"  Thought: {result.get('thought', 'N/A')}")
            print(f"  Tool: {result.get('tool', 'N/A')}")
            print(f"  Input: {result.get('input', 'N/A')}")
            
            if result.get("tool") == "final_answer":
                print(f"\nFinal Recommendation: {result['input']}")
                return result["input"]
            
            observation = self.simulate_tool(result["tool"], result["input"])
            print(f"  Observation: {observation}")
            messages.append({"role": "assistant", "content": response.choices[0].message.content})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        
        return "Max steps reached"

agent = ToolAgent()
agent.process("Should our PE client acquire a specialty chemicals company with $80M revenue and 22% EBITDA margins? What's the market outlook and how does the target compare to peers?")

---
## Summary

### Key Takeaways

1. **Zero-shot vs. Few-shot**: Few-shot prompting provides examples that guide the model toward consistent format and behavior. Use it when you need predictable, structured responses.

2. **Chain-of-Thought (CoT)**: Explicitly asking for step-by-step reasoning improves accuracy on complex tasks. CoT is especially valuable for math, logic, and multi-step problems.

3. **Role-Based Prompting**: System prompts are the primary mechanism for defining agent personas. A well-crafted system prompt shapes the entire behavior profile of an agent.

4. **Structured Outputs**: JSON mode ensures machine-readable responses, which is critical for agentic workflows where outputs feed into downstream code or other agents.

5. **Multi-Turn Management**: Maintaining conversation history enables context-aware interactions, but requires careful management of the context window as conversations grow.

6. **LangChain Prompt Templates**: LangChain's `ChatPromptTemplate` and `StructuredOutputParser` provide reusable, composable abstractions for prompt engineering. The pipe operator (`|`) lets you chain prompts, LLMs, and parsers into end-to-end workflows.

### Looking Ahead

These prompt engineering techniques form the foundation for building agentic systems. In the next session, we will explore how to evaluate and measure the quality of agent outputs systematically.

In [ ]:
# Task 4 Solution: Build a Multi-Step Reasoning Agent with Tool Descriptions

tools = [
    {"name": "calculator", "description": "Performs mathematical calculations", "parameters": "expression (string)"},
    {"name": "search", "description": "Searches for factual information", "parameters": "query (string)"},
    {"name": "weather", "description": "Gets current weather for a location", "parameters": "city (string)"}
]

class ToolAgent:
    def __init__(self):
        self.client = openai.OpenAI()
        tool_descriptions = "\n".join([f"- {t['name']}: {t['description']} (params: {t['parameters']})" for t in tools])
        self.system_prompt = f"""You are an AI agent with access to the following tools:

{tool_descriptions}

When you need to use a tool, respond with JSON:
{{"thought": "your reasoning", "tool": "tool_name", "input": "tool_input"}}

When you have enough information to answer, respond with JSON:
{{"thought": "final reasoning", "tool": "final_answer", "input": "your complete answer"}}

Process one step at a time."""
    
    def simulate_tool(self, tool_name, tool_input):
        simulated = {
            "calculator": f"Result: {tool_input} = [calculated value]",
            "search": f"Search results for '{tool_input}': [relevant information found]",
            "weather": f"Weather in {tool_input}: 72\u00b0F, partly cloudy"
        }
        return simulated.get(tool_name, "Tool not found")
    
    def process(self, question):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": question}
        ]
        
        for step in range(5):  # Max 5 steps
            response = self.client.chat.completions.create(
                model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
                messages=messages,
                response_format={"type": "json_object"},
                max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
            )
            result = json.loads(response.choices[0].message.content)
            print(f"\nStep {step + 1}:")
            print(f"  Thought: {result.get('thought', 'N/A')}")
            print(f"  Tool: {result.get('tool', 'N/A')}")
            print(f"  Input: {result.get('input', 'N/A')}")
            
            if result.get("tool") == "final_answer":
                print(f"\nFinal Answer: {result['input']}")
                return result["input"]
            
            observation = self.simulate_tool(result["tool"], result["input"])
            print(f"  Observation: {observation}")
            messages.append({"role": "assistant", "content": response.choices[0].message.content})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        
        return "Max steps reached"

agent = ToolAgent()
agent.process("What's 15% tip on a $85 dinner, and what's the weather like in New York?")

---
## Summary

### Key Takeaways

1. **Zero-shot vs. Few-shot**: Few-shot prompting provides examples that guide the model toward consistent format and behavior. Use it when you need predictable, structured responses.

2. **Chain-of-Thought (CoT)**: Explicitly asking for step-by-step reasoning improves accuracy on complex tasks. CoT is especially valuable for math, logic, and multi-step problems.

3. **Role-Based Prompting**: System prompts are the primary mechanism for defining agent personas. A well-crafted system prompt shapes the entire behavior profile of an agent.

4. **Structured Outputs**: JSON mode ensures machine-readable responses, which is critical for agentic workflows where outputs feed into downstream code or other agents.

5. **Multi-Turn Management**: Maintaining conversation history enables context-aware interactions, but requires careful management of the context window as conversations grow.

### Looking Ahead

These prompt engineering techniques form the foundation for building agentic systems. In the next session, we will explore how to evaluate and measure the quality of agent outputs systematically.